# D3: block ownership, reference counts and reuse

**Goal:** assign D2 block tables from a free list, grow request capacity, and reuse physical blocks after a request finishes.

**Spec:** no duplicate live ownership; blocks return only after their final reference; insufficient capacity fails without partial allocation; reservations do not commit tokens; reused storage matches separate FP32 ours/HF forwards. D2 manual tables and existing no-cache/contiguous paths remain supported.

Managed requests use **disjoint blocks**. Allocator reference counts above one are tested separately. No shared-prefix attention, copy-on-write, scheduler, preemption or fast kernel here.

Run top to bottom on a **fresh Colab T4**. Two FP32 TinyLlama copies need about 8.8 GB of weights. Runtime validation is pending; local checks are static only. Preserve previous outputs before replacing the runtime.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


## 1. Test first

Run B8/C2/C3/D1/D2/D3 tests. D3 includes 20 seeded allocator traces of 400 operations and ten managed-request traces of 200 operations. They check independent reference counts, disjoint tables, conservation, exhaustion, rounding, reset, release and reuse.

Small CPU/CUDA model tests compare managed cached logits with separate ours/HF and inject a later-layer failure before retry. A failed test stops this lesson. The C2 two-GPU test may skip on one T4; CUDA tests should run.

In [ ]:
import gc
import torch
from tinyserve.basics import load_model, encode
from tinyserve.engine.block_manager import BlockAllocator, BlockPoolExhausted
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.models.attention import AttentionMetadata
from tinyserve.probe import compare_paged_batch
from tinyserve.envinfo import collect_env, save_result

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b8_model.py",
                           "tests/test_c2_generation.py", "tests/test_c3_cache.py",
                           "tests/test_d1_batching.py", "tests/test_d2_paged_cache.py",
                           "tests/test_d3_block_manager.py", "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. A reference is a lease, not a token count

A three-block allocator starts with counts `[0, 0, 0]`. Allocate two blocks, retain block 0, then release it twice. The first release must not make it free; the final release must. Recently freed IDs are reused first by this stack free list.

`free_count + live_physical_block_count == pool_size`. The **sum** of references can exceed the live block count. Reference counts alone do not implement shared-prefix attention or copy-on-write.

In [ ]:
allocator = BlockAllocator(3)
allocator_trace = []

def record_allocator(label):
    assert set(allocator.free_blocks) == {index for index, refs in enumerate(allocator.ref_counts) if refs == 0}
    assert allocator.num_free_blocks + sum(refs > 0 for refs in allocator.ref_counts) == allocator.num_blocks
    row = {"label": label, "free_blocks": list(allocator.free_blocks), "ref_counts": list(allocator.ref_counts)}
    allocator_trace.append(row)
    print(row)

record_allocator("start")
assert allocator.allocate(2) == (0, 1)
record_allocator("allocate two")
allocator.retain(0)
record_allocator("retain zero")
allocator.release(0)
assert allocator.ref_counts[0] == 1 and 0 not in allocator.free_blocks
record_allocator("release one lease")
allocator.release(0)
record_allocator("release final lease")
assert allocator.allocate(1) == (0,)
record_allocator("reuse zero")
allocator.release(0)
allocator.release(1)
assert allocator.num_free_blocks == 3
record_allocator("all released")


## 3. Reserve capacity, then commit tokens

Create four physical blocks with B=2. Request 0 needs three token slots (two blocks); request 1 needs two slots (one block). One block remains free. The tensors are allocated once; assigning tables does not allocate GPU storage.

Use short slices of real tokenized prompts to make the block arithmetic exact. Later appended tokens also come from these fixed prompts; this is a cache-lifecycle demo, not a generation loop.

Reserve calls take **total processed-token capacity**, not append count. Forwarding uses the same D2 metadata/gather path. Each successful forward commits only active request lengths. FP32 assertions use `rtol=1e-5, atol=1e-4` plus exact all-row argmax checks.

In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
hf_model.set_attn_implementation("eager")
our_model = LlamaForCausalLM.from_huggingface(hf_model)
prompt_texts = ["The capital of France is", "The capital of Germany is",
                "A tiny language model can help people by"]
encoded_prompts = [encode(tokenizer, text, DEVICE) for text in prompt_texts]
assert all(ids.numel() >= 5 for ids in encoded_prompts)
BLOCK_SIZE, NUM_BLOCKS = 2, 4
managed_pool = our_model.new_managed_paged_cache(NUM_BLOCKS, BLOCK_SIZE)
managed_pool.key.fill_(float("nan"))
managed_pool.value.fill_(float("nan"))
pool_pointers = (managed_pool.key.data_ptr(), managed_pool.value.data_ptr())
stage_results, pool_trace = [], []

def pool_state():
    return {"tables": {str(owner): list(table) for owner, table in managed_pool.block_tables.items()},
            "lengths": {str(owner): length for owner, length in managed_pool.lengths.items()},
            "free_blocks": list(managed_pool.free_blocks), "ref_counts": list(managed_pool.ref_counts)}

def record_pool(label):
    used = [block for table in managed_pool.block_tables.values() for block in table]
    assert len(used) == len(set(used))
    assert set(used).isdisjoint(managed_pool.free_blocks)
    assert set(used).union(managed_pool.free_blocks) == set(range(NUM_BLOCKS))
    assert managed_pool.ref_counts == tuple(int(block in used) for block in range(NUM_BLOCKS))
    assert pool_pointers == (managed_pool.key.data_ptr(), managed_pool.value.data_ptr())
    row = {"label": label, **pool_state()}
    pool_trace.append(row)
    print(row)

def run_stage(label, active_requests):
    comparison = compare_paged_batch(our_model, hf_model, managed_pool, active_requests)
    stage_results.append({"label": label, "comparison": comparison})
    record_pool(label)
    print(comparison)

assert managed_pool.reserve(0, 3) == (0, 1)
assert managed_pool.reserve(1, 2) == (2,)
assert dict(managed_pool.lengths) == {0: 0, 1: 0}
record_pool("reserved, not committed")
run_stage("initial prefill", {0: encoded_prompts[0][:3], 1: encoded_prompts[1][:2]})
assert dict(managed_pool.lengths) == {0: 3, 1: 2}


## 4. Exhaustion is not permission to steal live blocks

Request 2 needs four token slots (two blocks), but only one block is free. Admission must raise without taking that last block or adding an empty request. Growth of request 0 to seven slots also needs two more blocks and must fail atomically.

Then reserve five slots for request 0. This adds the last free block but leaves its committed length at three. A scheduler could delay admission or preempt work; D3 deliberately does neither. Atomicity is per reservation, not across multiple reserve calls.

In [ ]:
exhaustion_results = []
for request_id, total_tokens in ((2, 4), (0, 7)):
    before = pool_state()
    try:
        managed_pool.reserve(request_id, total_tokens)
    except BlockPoolExhausted as error:
        row = {"request_id": request_id, "total_tokens": total_tokens, "error": str(error),
               "state_unchanged": pool_state() == before}
        assert row["state_unchanged"]
        exhaustion_results.append(row)
        print("expected exhaustion:", row)
    else:
        raise AssertionError("insufficient capacity must raise")
assert managed_pool.reserve(0, 5) == (0, 1, 3)
assert managed_pool.lengths[0] == 3 and managed_pool.num_free_blocks == 0
record_pool("grown across boundary, still length three")


## 5. A failed forward keeps its reservation for retry

Inject a failure before the second decoder layer. The first layer writes append slots for positions 3 and 4, but those tokens must not commit. Both K and V for the old prefix must stay unchanged in every layer. Always remove the failure hook before retry.

Retry the same append chunk against ours/HF references. Request 1 is paused and remains at length two.

In [ ]:
def prefix_bytes(request_id):
    positions = torch.arange(managed_pool.lengths[request_id], device=managed_pool.key.device)
    owners = positions.new_full(positions.shape, request_id)
    slots = managed_pool.slot_mapping(positions, owners).long()
    return tuple(storage.flatten(1, 2)[:, slots].clone()
                 for storage in (managed_pool.key, managed_pool.value))

prefix_before = {owner: prefix_bytes(owner) for owner in managed_pool.lengths}
before_failure = pool_state()

def injected_failure(module, args):
    raise RuntimeError("injected D3 later-layer failure")

failure_hook = our_model.model.layers[1].register_forward_pre_hook(injected_failure)
try:
    try:
        with torch.no_grad():
            append_ids = encoded_prompts[0][3:5]
            append_positions = torch.arange(3, 5, device=append_ids.device, dtype=append_ids.dtype)
            append_owners = torch.zeros_like(append_ids)
            our_model(append_ids, append_positions,
                      AttentionMetadata(append_positions, append_owners, paged_cache=managed_pool))
    except RuntimeError as error:
        if str(error) != "injected D3 later-layer failure":
            raise
        failure_error = str(error)
        print("expected forward failure:", failure_error)
    else:
        raise AssertionError("failure hook did not execute")
finally:
    failure_hook.remove()
assert pool_state() == before_failure
for owner, previous in prefix_before.items():
    for actual, expected in zip(prefix_bytes(owner), previous):
        torch.testing.assert_close(actual, expected, rtol=0, atol=0)
record_pool("failure preserved ownership and committed prefixes")
run_stage("retry boundary-crossing append", {0: encoded_prompts[0][:5]})
assert dict(managed_pool.lengths) == {0: 5, 1: 2}


## 6. Release, reuse and mixed work

Release request 0's three blocks. Admit request 2 with two of those IDs, then grow request 1 into the remaining block. Physical order is now scrambled automatically: no table sorting is allowed.

Forward a four-token prefill for request 2 with a one-token append for request 1. Old request 0 bytes remain in storage but must never become request 2's unwritten prefix. Pool tensor addresses and allocated bytes do not change.

Reset hides a prefix but keeps its reservation. Release removes ownership entirely. Neither operation securely erases bytes or returns tensor memory to CUDA.

In [ ]:
released_blocks = set(managed_pool.block_tables[0])
managed_pool.release_request(0)
assert 0 not in managed_pool.block_tables and 0 not in managed_pool.lengths
assert managed_pool.num_free_blocks == 3 and managed_pool.lengths[1] == 2
record_pool("request zero released")
managed_pool.reserve(2, 4)
assert set(managed_pool.block_tables[2]) <= released_blocks
reused_blocks = list(managed_pool.block_tables[2])
managed_pool.reserve(1, 3)
record_pool("reused blocks and grew remaining request")
run_stage("mixed new prefill and resumed append", {2: encoded_prompts[2][:4], 1: encoded_prompts[1][:3]})
assert dict(managed_pool.lengths) == {1: 3, 2: 4}
final_lengths_before_reset = dict(managed_pool.lengths)
reset_table = managed_pool.block_tables[2]
managed_pool.reset(2)
assert managed_pool.lengths[2] == 0 and managed_pool.lengths[1] == 3
assert managed_pool.block_tables[2] == reset_table and managed_pool.num_free_blocks == 0
run_stage("reset without release", {2: encoded_prompts[0][:2]})
for request_id in tuple(managed_pool.block_tables):
    managed_pool.release_request(request_id)
assert managed_pool.num_free_blocks == NUM_BLOCKS and managed_pool.ref_counts == (0,) * NUM_BLOCKS
assert dict(managed_pool.block_tables) == dict(managed_pool.lengths) == {}
record_pool("all requests released")


## 7. Save evidence and interview questions

1. **Why count references?** A physical block remains live until its last lease is released; token count is unrelated.
2. **Why separate reservation and commit?** Reserve before execution, then expose new tokens only after the full forward succeeds. Retry overwrites incomplete append slots.
3. **Does release reduce GPU allocation?** No. It recycles IDs inside a fixed tensor pool. Scheduling, prefix sharing and performance kernels are separate lessons.

Numerical difference ranges below are correctness evidence, not timings. Save the JSON and notebook with outputs before replacing the runtime. **Next after D3 validation: D4, continuous batching and preemption.**

In [ ]:
all_rows = [row for stage in stage_results for row in stage["comparison"]["requests"]]
no_cache_diffs = [row["paged_vs_no_cache_max_abs_diff"] for row in all_rows]
hf_diffs = [row["paged_vs_hf_max_abs_diff"] for row in all_rows]
saved_path = save_result("D3_block_manager", {
    "dtype": "float32", "backend": "pytorch_paged_gather", "managed_disjoint_blocks": True,
    "block_size": BLOCK_SIZE, "num_blocks": NUM_BLOCKS,
    "pool_shape": list(managed_pool.key.shape), "pool_allocated_bytes": managed_pool.allocated_bytes,
    "allocator_trace": allocator_trace, "pool_trace": pool_trace, "stages": stage_results,
    "exhaustion": exhaustion_results, "forward_failure_error": failure_error,
    "failed_forward_preserved_metadata_and_prefix_kv": True,
    "reused_blocks": reused_blocks, "final_lengths_before_reset": final_lengths_before_reset,
    "storage_addresses_unchanged": pool_pointers == (managed_pool.key.data_ptr(), managed_pool.value.data_ptr()),
    "all_blocks_released": managed_pool.num_free_blocks == NUM_BLOCKS,
    "paged_vs_no_cache_diff_range": {"min": min(no_cache_diffs), "max": max(no_cache_diffs)},
    "paged_vs_hf_diff_range": {"min": min(hf_diffs), "max": max(hf_diffs)},
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


In [ ]:
del managed_pool, prefix_before, previous, actual, expected, our_model, hf_model, failure_hook
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
